# threading

In [ ]:
///- --test static

open testing

In [ ]:
open rust
open rust_operators

## rust

### join_handle

In [ ]:
nominal join_handle t =
    `(
        rust_type
            `($'std_thread_JoinHandle<`t>')
            `($'std::thread::JoinHandle<`t>')
            "type std_thread_JoinHandle<'T> = class end"
    )

### spawn

In [ ]:
inl spawn forall t. depth flag (x : () -> t) : join_handle t =
    if flag = 1u8
    then (!\($'"true; let __spawn = std::thread::spawn(move || { //"') : bool) |> ignore
    else (!\($'"true; let __spawn = std::thread::spawn(|| { //"') : bool) |> ignore

    let x' = x ()
    inl x' = join x'

    x' |> rust.fix_closure depth

    !\($'"__spawn"')

### join'

In [ ]:
inl join' forall t.
    (x : join_handle t)
    : resultm.result'
        t
        (
            rust.box (
                rust.lifetime_ref
                    rust.dyn'
                    (
                        rust.lifetime_join
                            rust.any
                            (
                                rust.lifetime_ref
                                    rust.send
                                    rust.static_lifetime
                            )
                    )
            )
        ) =
    !\\(x, $'"std::thread::JoinHandle::join($0)"')

### arc

In [ ]:
nominal arc t =
    `(
        rust_type
            `($'std_sync_Arc<`t>')
            `($'std::sync::Arc<`t>')
            "type std_sync_Arc<'T> = class end"
    )

### new_arc

In [ ]:
inl new_arc forall t. (x : t) : arc t =
    !\($'"std::sync::Arc::new(!x)"')

### mutex

In [ ]:
nominal mutex t =
    `(
        rust_type
            `($'std_sync_Mutex<`t>')
            `($'std::sync::Mutex<`t>')
            "type std_sync_Mutex<'T> = class end"
    )

### new_mutex

In [ ]:
inl new_mutex forall t. (x : t) : mutex t =
    !\($'"std::sync::Mutex::new(!x)"')

### new_mutex_mut

In [ ]:
inl new_mutex_mut forall t. (x : rust.ref (rust.mut' t)) : mutex t =
    !\($'"std::sync::Mutex::new(!x)"')

### rw_lock

In [ ]:
nominal rw_lock t =
    `(
        rust_type
            `($'std_sync_RwLock<`t>')
            `($'std::sync::RwLock<`t>')
            "type std_sync_RwLock<'T> = class end"
    )

### new_rw_lock

In [ ]:
inl new_rw_lock forall t. (x : t) : rw_lock t =
    !\\(x, $'"std::sync::RwLock::new($0)"')

### new_arc_mutex

In [ ]:
inl new_arc_mutex forall t. (x : t) : arc (mutex t) =
    x |> new_mutex |> new_arc

### new_arc_rw_lock

In [ ]:
inl new_arc_rw_lock forall t. (x : t) : arc (rw_lock t) =
    x |> new_rw_lock |> new_arc

### arc_clone

In [ ]:
inl arc_clone forall t. (x : arc t) : arc t =
    inl x = join x
    !\($'"std::sync::Arc::clone(&!x)"')

### arc_ptr_eq

In [ ]:
inl arc_ptr_eq forall t. (a : rust.ref (arc t)) (b : rust.ref (arc t)) : bool =
    !\\((a, b), $'"std::sync::Arc::ptr_eq($0, $1)"')

### arc_try_unwrap

In [ ]:
inl arc_try_unwrap forall t. (x : arc t) : resultm.result' t (arc t) =
    !\\(x, $'"std::sync::Arc::try_unwrap($0)"')

### arc_into_raw

In [ ]:
inl arc_into_raw forall t. (x : arc t) : rust.ptr t =
    !\\(x, $'"std::sync::Arc::into_raw($0)"')

### arc_from_raw

In [ ]:
inl arc_from_raw forall t. (x : rust.ptr t) : arc t =
    !\\(x, $'"std::sync::Arc::from_raw($0)"')

### partial_eq_wrapper_arc_eq

In [ ]:
inl partial_eq_wrapper_arc_eq forall t.
    (self : rust.ref (rust.partial_eq_wrapper (arc t)))
    (other : rust.ref (rust.partial_eq_wrapper (arc t)))
    =
    self
    |> rust.unwrap_0_ref
    |> arc_ptr_eq (
        other
        |> rust.unwrap_0_ref
    )

### new_partial_eq_wrapper_arc

In [ ]:
inl new_partial_eq_wrapper_arc forall t. (x : arc t) : rust.partial_eq_wrapper (arc t) =
    x |> rust.new_partial_eq_wrapper partial_eq_wrapper_arc_eq

### mutex_guard

In [ ]:
nominal mutex_guard t =
    `(
        rust_type
            `($'std_sync_MutexGuard<`t>')
            `($'std::sync::MutexGuard<`t>')
            "type std_sync_MutexGuard<'T> = class end"
    )

### rw_lock_read_guard

In [ ]:
nominal rw_lock_read_guard t =
    `(
        rust_type
            `($'std_sync_RwLockReadGuard<`t>')
            `($'std::sync::RwLockReadGuard<`t>')
            "type std_sync_RwLockReadGuard<'T> = class end"
    )

### rw_lock_write_guard

In [ ]:
nominal rw_lock_write_guard t =
    `(
        rust_type
            `($'std_sync_RwLockWriteGuard<`t>')
            `($'std::sync::RwLockWriteGuard<`t>')
            "type std_sync_RwLockWriteGuard<'T> = class end"
    )

### poison_error

In [ ]:
nominal poison_error t =
    `(
        rust_type
            `($'std_sync_PoisonError<`t>')
            `($'std::sync::PoisonError<`t>')
            "type std_sync_PoisonError<'T> = class end"
    )

### try_lock_error

In [ ]:
nominal try_lock_error t =
    `(
        rust_type
            `($'std_sync_TryLockError<`t>')
            `($'std::sync::TryLockError<`t>')
            "type std_sync_TryLockError<'T> = class end"
    )

### arc_mutex_lock

In [ ]:
inl arc_mutex_lock forall t. (x : arc (mutex t)) : resultm.result' (mutex_guard t) (poison_error (mutex_guard t)) =
    inl x = x |> rust.emit
    !\($'"!x.lock()"')

### arc_mutex_lock'

In [ ]:
inl arc_mutex_lock' forall t. (x : arc (mutex t)) : resultm.result' (mutex_guard t) (poison_error (mutex_guard t)) =
    !\\(x, $'"$0.lock()"')

### arc_rw_lock_read

In [ ]:
inl arc_rw_lock_read forall t. (x : arc (rw_lock t)) : resultm.result' (rw_lock_read_guard t) (poison_error (rw_lock_read_guard t)) =
    inl x = x |> rust.emit
    !\($'"!x.read()"')

### arc_rw_lock_write

In [ ]:
inl arc_rw_lock_write forall t. (x : arc (rw_lock t)) : resultm.result' (rw_lock_write_guard t) (poison_error (rw_lock_write_guard t)) =
    inl x = x |> rust.emit
    !\($'"!x.write()"')

### arc_rw_lock_try_read

In [ ]:
inl arc_rw_lock_try_read forall t. (x : arc (rw_lock t)) : resultm.result' (rw_lock_read_guard t) (try_lock_error (rw_lock_read_guard t)) =
    inl x = x |> rust.emit
    !\($'"!x.try_read()"')

### arc_rw_lock_try_write

In [ ]:
inl arc_rw_lock_try_write forall t. (x : arc (rw_lock t)) : resultm.result' (rw_lock_write_guard t) (try_lock_error (rw_lock_write_guard t)) =
    inl x = x |> rust.emit
    !\($'"!x.try_write()"')

### mutex_guard_ref

In [ ]:
inl mutex_guard_ref forall t. (x : mutex_guard t) : rust.ref t =
    !\\(x, $'"&$0"')

### rw_lock_read_guard_ref

In [ ]:
inl rw_lock_read_guard_ref forall t. (x : rw_lock_read_guard t) : rust.ref t =
    !\\(x, $'"&$0"')

### rw_lock_write_guard_ref

In [ ]:
inl rw_lock_write_guard_ref forall t. (x : rw_lock_write_guard t) : rust.ref t =
    !\\(x, $'"&$0"')

### mutex_guard_ref_mut

In [ ]:
inl mutex_guard_ref_mut forall t. (x : mutex_guard t) : rust.ref (rust.mut' t) =
    inl x = join x
    (!\($'"true; let mut !x = !x"') : bool) |> ignore
    !\\(x, $'"&mut $0"')

### mutex_guard_ref_mut''

In [ ]:
inl mutex_guard_ref_mut'' forall t. (x : mutex_guard t) : rust.ref (rust.mut' t) =
    inl x = x |> rust.emit
    (!\($'"true; let mut !x = !x"') : bool) |> ignore
    !\\(x, $'"&mut $0"')

### mutex_guard_ref_mut'

In [ ]:
inl mutex_guard_ref_mut' forall t. (x : mutex_guard (rust.ref (rust.mut' t))) : rust.ref (rust.mut' t) =
    inl x = x |> rust.emit
    (!\($'"true; let mut !x = !x"') : bool) |> ignore
    !\\(x, $'"&mut $0"')

### mutex_guard_as_mut

In [ ]:
inl mutex_guard_as_mut forall (t : * -> *) u. (x : mutex_guard (t u)) : t (rust.ref (rust.mut' u)) =
    (!\($'"true; let mut !x = !x"') : bool) |> ignore
    !\\(x, $'"$0.as_mut()"')

### channel_receiver

In [ ]:
nominal channel_receiver t =
    `(
        rust_type
            `($'std_sync_mpsc_Receiver<`t>')
            `($'std::sync::mpsc::Receiver<`t>')
            "type std_sync_mpsc_Receiver<'T> = class end"
    )

### channel_sender

In [ ]:
nominal channel_sender t =
    `(
        rust_type
            `($'std_sync_mpsc_Sender<`t>')
            `($'std::sync::mpsc::Sender<`t>')
            "type std_sync_mpsc_Sender<'T> = class end"
    )

### new_channel

In [ ]:
inl new_channel () : channel_sender sm'.std_string * arc (channel_receiver sm'.std_string) =
    !\($'"{ let (sender, receiver) = std::sync::mpsc::channel(); (sender, std::sync::Arc::new(receiver)) }"')

### send_error

In [ ]:
nominal send_error t =
    `(
        rust_type
            `($'std_sync_mpsc_SendError<`t>')
            `($'std::sync::mpsc::SendError<`t>')
            "type std_sync_mpsc_SendError<'T> = class end"
    )

### channel_send

In [ ]:
inl channel_send forall t. (line : t) (sender : rust.ref (channel_sender t)) : resultm.result' () (send_error sm'.std_string) =
    !\\((sender, line), $'"$0.send($1)"')

## fsharp

### sleep

In [ ]:
inl sleep (n : i32) : () =
    run_target function
        | Fsharp => fun () => $'System.Threading.Thread.Sleep' n
        | Rust => fun () => $'std::thread::sleep(std::time::Duration::from_millis(!n as u64))' : ()
        | _ => fun () => ()

### thread

In [ ]:
nominal thread = $'System.Threading.Thread'

### cancellation_token

In [ ]:
type cancellation_token_switch =
    {
        Gleam : $'System.Threading.CancellationToken'
        Lua : $'System.Threading.CancellationToken'
        Fsharp : $'System.Threading.CancellationToken'
        Python : $'System.Threading.CancellationToken'
        TypeScript : $'System.Threading.CancellationToken'
        Cpp : $'System.Threading.CancellationToken'
        CppHost : $'System.Threading.CancellationToken'
        Delphi : $'System.Threading.CancellationToken'
        Cuda : $'System.Threading.CancellationToken'
        C : $'System.Threading.CancellationToken'
        Rust : $'std::sync::Arc<std::sync::atomic::AtomicBool>'
    }
nominal cancellation_token = $'backend_switch `(cancellation_token_switch)'

### cancellation_token_source

In [ ]:
type cancellation_token_source_switch =
    {
        Gleam : $'System.Threading.CancellationTokenSource'
        Lua : $'System.Threading.CancellationTokenSource'
        Fsharp : $'System.Threading.CancellationTokenSource'
        Python : $'System.Threading.CancellationTokenSource'
        TypeScript : $'System.Threading.CancellationTokenSource'
        Cpp : $'System.Threading.CancellationTokenSource'
        CppHost : $'System.Threading.CancellationTokenSource'
        Delphi : $'System.Threading.CancellationTokenSource'
        Cuda : $'System.Threading.CancellationTokenSource'
        C : $'System.Threading.CancellationTokenSource'
        Rust : $'std::sync::Arc<std::sync::atomic::AtomicBool>'
    }
nominal cancellation_token_source = $'backend_switch `(cancellation_token_source_switch)'

### cancellation_token_registration

In [ ]:
type cancellation_token_registration_switch =
    {
        Gleam : $'System.Threading.CancellationTokenRegistration'
        Lua : $'System.Threading.CancellationTokenRegistration'
        Fsharp : $'System.Threading.CancellationTokenRegistration'
        Python : $'System.Threading.CancellationTokenRegistration'
        TypeScript : $'System.Threading.CancellationTokenRegistration'
        Cpp : $'System.Threading.CancellationTokenRegistration'
        CppHost : $'System.Threading.CancellationTokenRegistration'
        Delphi : $'System.Threading.CancellationTokenRegistration'
        Cuda : $'System.Threading.CancellationTokenRegistration'
        C : $'System.Threading.CancellationTokenRegistration'
        Rust : $'()'
    }
nominal cancellation_token_registration = $'backend_switch `(cancellation_token_registration_switch)'

### cancellation_source_token

In [ ]:
inl cancellation_source_token (x : cancellation_token_source) : cancellation_token =
    if is_rust ()
    then $'!x.clone()'
    else $'!x.Token'

### cancellation_source_cancel

In [ ]:
inl cancellation_source_cancel (x : cancellation_token_source) : () =
    run_target function
        | Fsharp => fun () =>
            $'!x.Cancel' ()
        | Rust => fun () => $'!x.store(true, std::sync::atomic::Ordering::SeqCst)' : ()
        | _ => fun () => null ()

### create_linked_token_source

In [ ]:
inl create_linked_token_source (x : array_base cancellation_token) : cancellation_token_source =
    x |> $'System.Threading.CancellationTokenSource.CreateLinkedTokenSource'

### concurrent_stack

In [ ]:
nominal concurrent_stack t = $'System.Collections.Concurrent.ConcurrentStack<`t>'

### concurrent_stack_push

In [ ]:
inl concurrent_stack_push forall t. (item : t) (stack : concurrent_stack t) : () =
    run_target function
        | Fsharp => fun () => $'!stack.Push' item
        | _ => fun () => ()

### token_none

In [ ]:
inl token_none () : cancellation_token =
    if is_rust ()
    then $'std::sync::Arc::new(std::sync::atomic::AtomicBool::new(false))'
    else $'`cancellation_token.None'

### new_concurrent_stack

In [ ]:
inl new_concurrent_stack forall t. () : concurrent_stack t =
    $'System.Collections.Concurrent.ConcurrentStack<`t>' ()

### token_register

In [ ]:
inl token_register (fn : () -> ()) (ct : cancellation_token) : cancellation_token_registration =
    fn |> $'!ct.Register'

### new_cancellation_token_source

In [ ]:
inl new_cancellation_token_source () : cancellation_token_source =
    if is_rust ()
    then $'std::sync::Arc::new(std::sync::atomic::AtomicBool::new(false))'
    else $'new `cancellation_token_source ()'

inl token_cancellation_requested (ct : cancellation_token) : bool =
    if is_rust ()
    then $'!ct.load(std::sync::atomic::Ordering::SeqCst)'
    else $'!ct.IsCancellationRequested'

### new_disposable_token

In [ ]:
inl new_disposable_token (merge_token : optionm'.option' cancellation_token) =
    run_target function
        | Fsharp => fun () =>
            inl cts = new_cancellation_token_source ()
            inl cts =
                match merge_token |> optionm'.unbox with
                | None => cts
                | Some merge_token =>
                    create_linked_token_source ;[ cts |> cancellation_source_token; merge_token ]
            inl disposable : _ () = new_disposable fun () =>
                cts |> cancellation_source_cancel
            cts |> cancellation_source_token, disposable
        | _ => fun () => null ()

In [1]:
///- --test
///> fsharp

inl run fn =
    inl token, disposable = new_disposable_token (None |> optionm'.box)
    fn token
    $'!disposable.Dispose()'
    fun () =>
        fn token
    |> async.new_task
    |> async.await_task
    |> async.start

fun () =>
    inl counter = mut 0i32

    inl fn (token : cancellation_token) =
        counter <- *counter + (if token |> token_cancellation_requested then 10 else 1)

    join run fn
    async.sleep 10 |> async.do
    return_in *counter
|> async.new_task
|> fun x => x : _ int
|> async.await_task
|> async.run_synchronously
|> _assert_eq 11i32

__assert_eq { expected = 11 }

## main

In [2]:
///> fsharp

inl main () =
    $'let new_disposable_token x = !new_disposable_token x' : ()